# PART 1 — Environment & Data Setup

This notebook analyzes the Chicago Crime dataset using PySpark on AWS EMR.
The objective is to clean, transform, aggregate, and optimize large-scale crime data using distributed Spark processing.

In [1]:
%%configure -f
{
  "driverCores": 1,
  "driverMemory": "2g",
  "executorCores": 1,
  "executorMemory": "2g",
  "numExecutors": 1,
  "conf": {
    "spark.dynamicAllocation.enabled": "false"
  }
}

In [2]:
from pyspark.sql import functions as F

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "s3://<your-bucket>/shared/datasets/chicago_crime/crime/"
    )
)

VBox()

Starting Spark application


ID,Kind,State,Spark UI,Driver log,User,Current session?
0,pyspark,idle,""" application-id="""">Link",,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# PART 2 — Data Exploration

This section explores schema structure, data volume, and data quality characteristics.

In [3]:
print(df.columns)

total_rows = df.count()
print(f"Row count: {total_rows:,}")

df.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']
Row count: 8,548,383
+--------+-----------+----------------------+---------------------+----+-----------------------+-----------------------------+---------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                |IUCR|Primary Type           |Description                  |Location Description       |Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+-----------

In [4]:
numeric_types = ('int', 'bigint', 'double', 'float', 'long', 'integer')
timestamp_types = ('timestamp', 'date')
categorical_types = ('string', 'boolean')

numeric_cols = []
timestamp_cols = []
categorical_cols = []

for field in df.schema.fields:

    dtype = field.dataType.simpleString()

    if dtype.startswith("decimal") or dtype in numeric_types:
        numeric_cols.append(field.name)

    elif dtype in timestamp_types:
        timestamp_cols.append(field.name)

    elif dtype in categorical_types:
        categorical_cols.append(field.name)

print("Numeric columns:")
print(numeric_cols)

print("\nTimestamp columns:")
print(timestamp_cols)

print("\nCategorical columns:")
print(categorical_cols)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Numeric columns:
['ID', 'Beat', 'District', 'Ward', 'Community Area', 'X Coordinate', 'Y Coordinate', 'Year', 'Latitude', 'Longitude']

Timestamp columns:
[]

Categorical columns:
['Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'FBI Code', 'Updated On', 'Location']

In [5]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward  |Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0  |0          |0   |0    |0   |0           |0          |16012               |0     |0       |0   |47      |614814|613725        |0       |96560       |96560       |0   |0         |96560   |96560    |96560   |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

In [6]:
null_data = null_counts.collect()[0]

for c in df.columns:

    null_count = null_data[c]
    null_percent = (null_count / total_rows) * 100

    print(f"{c}: {null_percent:.2f}% nulls")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

ID: 0.00% nulls
Case Number: 0.00% nulls
Date: 0.00% nulls
Block: 0.00% nulls
IUCR: 0.00% nulls
Primary Type: 0.00% nulls
Description: 0.00% nulls
Location Description: 0.19% nulls
Arrest: 0.00% nulls
Domestic: 0.00% nulls
Beat: 0.00% nulls
District: 0.00% nulls
Ward: 7.19% nulls
Community Area: 7.18% nulls
FBI Code: 0.00% nulls
X Coordinate: 1.13% nulls
Y Coordinate: 1.13% nulls
Year: 0.00% nulls
Updated On: 0.00% nulls
Latitude: 1.13% nulls
Longitude: 1.13% nulls
Location: 1.13% nulls

In [7]:
df.select(
    "District",
    "Latitude"
).summary().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+------------------+------------------+
|summary|          District|          Latitude|
+-------+------------------+------------------+
|  count|           8548336|           8451823|
|   mean|11.295218975950407|41.842591086573385|
| stddev| 6.965374623067159|0.0891852166085482|
|    min|                 1|      36.619446395|
|    25%|                 6|        41.7690002|
|    50%|                10|      41.856694073|
|    75%|                17|      41.907104235|
|    max|                31|      42.022910333|
+-------+------------------+------------------+

# Observations

- The dataset contains more than 8 million records and requires distributed processing.
- Ward and Community Area contain the highest number of missing values.
- Geolocation columns contain null values, which can impact spatial analysis.
- The dataset contains categorical, temporal, and numerical features suitable for analytical workloads.

# Production Data Quality Concerns

1. Missing geolocation coordinates may reduce analytical accuracy for hotspot detection.
2. Invalid or malformed timestamps can break temporal analysis.
3. Duplicate Case Numbers or IDs can inflate crime statistics and aggregation results.

# PART 3 — Filtering & Cleaning

This section creates a cleaned dataframe called df_clean by applying integrity checks, deduplication, and data type conversions.

In [8]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    current_date,
    datediff
)

df_clean = df

df_clean = df_clean.filter(
    col("Latitude").isNotNull() &
    col("Longitude").isNotNull()
)

print(f"Rows after geolocation filtering: {df_clean.count():,}")

df_clean = df_clean.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

df_clean = df_clean.filter(
    col("Date").isNotNull()
)

print(f"Rows after temporal filtering: {df_clean.count():,}")

df_clean = df_clean.dropDuplicates(["ID"])
df_clean = df_clean.dropDuplicates(["Case Number"])

print(f"Rows after deduplication: {df_clean.count():,}")

df_clean = (
    df_clean
    .withColumn("Arrest", col("Arrest").cast("boolean"))
    .withColumn("Domestic", col("Domestic").cast("boolean"))
)

df_clean = df_clean.withColumn(
    "days_since_crime",
    datediff(current_date(), col("Date"))
)

df_clean.select(
    "Date",
    "days_since_crime"
).show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Rows after geolocation filtering: 8,451,823
Rows after temporal filtering: 8,451,823
Rows after deduplication: 8,451,207
+-------------------+----------------+
|Date               |days_since_crime|
+-------------------+----------------+
|2004-12-23 10:45:00|7824            |
|2001-01-05 04:27:00|9272            |
|2001-01-21 01:00:00|9256            |
|2001-01-27 11:45:49|9250            |
|2001-01-29 13:00:00|9248            |
+-------------------+----------------+
only showing top 5 rows

# PART 4 — Feature Engineering

This section creates additional analytical features using native Spark SQL functions.

In [9]:
from pyspark.sql.functions import (
    col,
    hour,
    when
)

df_clean = df_clean.withColumn(
    "time_of_day",
    when((hour(col("Date")) >= 0) & (hour(col("Date")) < 6), "Night")
    .when((hour(col("Date")) >= 6) & (hour(col("Date")) < 12), "Morning")
    .when((hour(col("Date")) >= 12) & (hour(col("Date")) < 18), "Afternoon")
    .otherwise("Evening")
)

print("Crime distribution by time of day:")

(df_clean.groupBy("time_of_day")
    .count()
    .orderBy("count", ascending=False)
    .show())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Crime distribution by time of day:
+-----------+-------+
|time_of_day|  count|
+-----------+-------+
|    Evening|2683755|
|  Afternoon|2625301|
|    Morning|1711477|
|      Night|1430674|
+-----------+-------+

In [10]:
violent_crimes = [
    "HOMICIDE",
    "CRIMINAL SEXUAL ASSAULT",
    "BATTERY"
]

df_clean = df_clean.withColumn(
    "is_violent_crime",
    when(
        col("Primary Type").isin(violent_crimes),
        True
    ).otherwise(False)
)

violent_crimes_count = df_clean.filter(
    col("is_violent_crime") == True
).count()

print(f"Number of violent crimes found: {violent_crimes_count:,}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of violent crimes found: 1,574,913

In [11]:
district_mapping = {
    1: "Central",
    2: "Wentworth",
    3: "Grand Crossing",
    4: "South Chicago",
    5: "Calumet",
    6: "Gresham",
    7: "Englewood",
    8: "Chicago Lawn",
    9: "Deering",
    10: "Ogden",
    11: "Harrison",
    12: "Near West",
    14: "Shakespeare",
    15: "Austin",
    16: "Jefferson Park",
    17: "Albany Park",
    18: "Near North",
    19: "Town Hall",
    20: "Lincoln",
    22: "Morgan Park",
    24: "Rogers Park",
    25: "Grand Central"
}

expr = when(
    col("District") == list(district_mapping.keys())[0],
    list(district_mapping.values())[0]
)

for district_id, district_name in list(district_mapping.items())[1:]:

    expr = expr.when(
        col("District") == district_id,
        district_name
    )

expr = expr.otherwise("Unknown")

df_clean = df_clean.withColumn(
    "district_name",
    expr
)

df_clean.select(
    "District",
    "district_name"
).distinct().show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+--------------+
|District|district_name |
+--------+--------------+
|22      |Morgan Park   |
|2       |Wentworth     |
|17      |Albany Park   |
|25      |Grand Central |
|16      |Jefferson Park|
|1       |Central       |
|6       |Gresham       |
|11      |Harrison      |
|9       |Deering       |
|15      |Austin        |
+--------+--------------+
only showing top 10 rows

In [12]:
df_clean.write.mode("overwrite").parquet(
    "s3://<your-bucket>/shared/user_artur/df_clean_parquet/"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

The cleaned dataset was persisted in Parquet format to avoid recomputing expensive transformations after session restarts.

# PART 5 — Aggregation & Join

This section aggregates crime statistics by district and calculates arrest effectiveness metrics.

In [13]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    round
)

df_district_metrics = df_clean.groupBy(
    "District",
    "district_name"
).agg(
    count("ID").alias("total_crimes"),
    sum(col("Arrest").cast("int")).alias("total_arrests")
)

df_district_metrics = df_district_metrics.withColumn(
    "arrest_rate",
    round(
        col("total_arrests") / col("total_crimes"),
        4
    )
)

df_final_report = df_district_metrics.orderBy(
    col("total_crimes").desc()
)

df_top_10_districts = df_final_report.limit(10)

df_top_10_districts.select(
    "District",
    "district_name",
    "total_crimes",
    "total_arrests",
    "arrest_rate"
).show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+--------------+------------+-------------+-----------+
|District|district_name |total_crimes|total_arrests|arrest_rate|
+--------+--------------+------------+-------------+-----------+
|8       |Chicago Lawn  |567406      |123664       |0.2179     |
|11      |Harrison      |533213      |216863       |0.4067     |
|6       |Gresham       |494588      |122181       |0.247      |
|4       |South Chicago |479320      |102778       |0.2144     |
|7       |Englewood     |477706      |130039       |0.2722     |
|25      |Grand Central |476706      |124672       |0.2615     |
|3       |Grand Crossing|430089      |104666       |0.2434     |
|12      |Near West     |427728      |91191        |0.2132     |
|9       |Deering       |410251      |109240       |0.2663     |
|2       |Wentworth     |403350      |99078        |0.2456     |
+--------+--------------+------------+-------------+-----------+

# PART 6 — Temporal Analysis (Window Functions)

This section analyzes crime trends over time using Spark window functions.

In [14]:
from pyspark.sql.functions import (
    col,
    year,
    month,
    count,
    dense_rank,
    lag,
    datediff
)

from pyspark.sql.window import Window

district_time_window = Window.partitionBy(
    "District"
).orderBy(
    col("Date")
)

df_temporal_analysis = df_clean.withColumn(
    "previous_crime_date",
    lag(col("Date"), 1).over(district_time_window)
)

df_final_temporal = df_temporal_analysis.withColumn(
    "days_since_last_crime",
    datediff(
        col("Date"),
        col("previous_crime_date")
    )
)

df_final_temporal.filter(
    col("District") == 1
).select(
    "District",
    "Date",
    "previous_crime_date",
    "days_since_last_crime"
).show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+-------------------+-------------------+---------------------+
|District|Date               |previous_crime_date|days_since_last_crime|
+--------+-------------------+-------------------+---------------------+
|1       |2001-01-01 00:00:00|NULL               |NULL                 |
|1       |2001-01-01 00:00:00|2001-01-01 00:00:00|0                    |
|1       |2001-01-01 00:01:00|2001-01-01 00:00:00|0                    |
|1       |2001-01-01 00:01:00|2001-01-01 00:01:00|0                    |
|1       |2001-01-01 00:01:00|2001-01-01 00:01:00|0                    |
|1       |2001-01-01 00:25:00|2001-01-01 00:01:00|0                    |
|1       |2001-01-01 00:30:00|2001-01-01 00:25:00|0                    |
|1       |2001-01-01 01:30:00|2001-01-01 00:30:00|0                    |
|1       |2001-01-01 02:30:00|2001-01-01 01:30:00|0                    |
|1       |2001-01-01 03:49:15|2001-01-01 02:30:00|0                    |
+--------+-------------------+-------------------+-

In [15]:
df_monthly_counts = df_clean.groupBy(
    year(col("Date")).alias("year"),
    month(col("Date")).alias("month"),
    col("District")
).agg(
    count("ID").alias("total_crimes")
)

ranking_window = Window.partitionBy(
    "year",
    "month"
).orderBy(
    col("total_crimes").desc()
)

df_ranked_districts = df_monthly_counts.withColumn(
    "monthly_crime_rank",
    dense_rank().over(ranking_window)
)

df_ranked_districts.filter(
    col("monthly_crime_rank") <= 3
).orderBy(
    "year",
    "month",
    "monthly_crime_rank"
).show(15, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+--------+------------+------------------+
|year|month|District|total_crimes|monthly_crime_rank|
+----+-----+--------+------------+------------------+
|2001|1    |2       |3076        |1                 |
|2001|1    |8       |2427        |2                 |
|2001|1    |25      |2207        |3                 |
|2001|2    |2       |2509        |1                 |
|2001|2    |8       |2144        |2                 |
|2001|2    |11      |1923        |3                 |
|2001|3    |2       |3070        |1                 |
|2001|3    |8       |2656        |2                 |
|2001|3    |11      |2358        |3                 |
|2001|4    |2       |3008        |1                 |
|2001|4    |8       |2569        |2                 |
|2001|4    |11      |2303        |3                 |
|2001|5    |2       |3047        |1                 |
|2001|5    |8       |2562        |2                 |
|2001|5    |11      |2341        |3                 |
+----+-----+--------+-------

# PART 7 — Production-Grade Output & Validation

The final dataset is written in optimized Parquet format and partitioned for efficient downstream analytics.

In [18]:
df_clean = spark.read.parquet(
    "s3://<your-bucket>/shared/user_artur/df_clean_parquet/"
)

df_to_write = df_clean.withColumnRenamed(
    "Primary Type",
    "Primary_Type"
)

output_s3_path = (
    "s3://<your-bucket>/"
    "shared/user_artur/chicago_crimes_enriched/"
)

df_to_write.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_s3_path)

print(
    f"Data successfully written to: {output_s3_path}"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Data successfully written to: s3://<your-bucket>/shared/user_artur/chicago_crimes_enriched/

# Predicate Pushdown and Partitioning

Predicate pushdown allows Spark to filter data earlier during file scanning instead of loading all records into memory.

Partitioning the dataset by Year improves query performance because Spark can skip unnecessary partitions when analysts filter by a specific year.

For example:
- filtering Year = 2024 only scans 2024 partitions
- older yearly partitions are ignored
- less disk I/O improves performance significantly

# PART 8 — Data Skew and Performance Tuning

In [19]:
(df_clean.groupBy("Primary Type")
    .count()
    .orderBy(col("count").desc())
    .show(5, truncate=False))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-------+
|Primary Type   |count  |
+---------------+-------+
|THEFT          |1794209|
|BATTERY        |1550111|
|CRIMINAL DAMAGE|966664 |
|NARCOTICS      |754430 |
|ASSAULT        |572683 |
+---------------+-------+
only showing top 5 rows

# Data Skew Explanation

If one crime category contains more than 50% of the records, Spark workers processing that partition may receive significantly more data than others.

This imbalance is called data skew.

Data skew can lead to:
- slow aggregations
- executor imbalance
- memory pressure
- larger shuffle operations
- delayed job completion

In [20]:
df_repartitioned = df_clean.repartition(
    500,
    "District"
)

print(
    f"Number of partitions: "
    f"{df_repartitioned.rdd.getNumPartitions()}"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of partitions: 500

# Repartitioning Trade-Off

Repartitioning improves parallelism by distributing data across more Spark partitions.

Advantages:
- better workload distribution
- improved parallel execution
- reduced bottlenecks

Trade-Off:
- repartition() triggers a full shuffle operation
- shuffling is expensive because data moves across executors
- excessive repartitioning increases network overhead

# coalesce() vs repartition()

# repartition()
- performs a full shuffle
- can increase or decrease partitions
- better data balancing
- more expensive operation

Use repartition() when:
- increasing partitions
- correcting partition imbalance
- improving distributed processing

# coalesce()
- minimizes data movement
- reduces partitions only
- more efficient for shrinking datasets

Use coalesce() when:
- reducing output files
- preparing final output writes
- avoiding unnecessary shuffle operations

# PART 9 — Data Quality and Schema Evolution

In [21]:
df_invalid_records = df_district_metrics.filter(
    col("arrest_rate") < 0.0
)

invalid_count = df_invalid_records.count()

if invalid_count > 0:

    df_invalid_records.show(5)

    raise ValueError(
        f"DATA QUALITY FAILURE: "
        f"Found {invalid_count} records with "
        f"a negative Arrest Rate."
    )

else:

    print(
        "DATA QUALITY PASSED: "
        "No negative Arrest Rate values found."
    )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DATA QUALITY PASSED: No negative Arrest Rate values found.

In [22]:
df_schema_evolved = df_clean.withColumnRenamed(
    "District",
    "Area_Code"
)

df_schema_evolved.select(
    "Area_Code"
).show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+
|Area_Code|
+---------+
|       19|
|        3|
|       14|
|        5|
|        4|
+---------+
only showing top 5 rows

# Why Column Renaming is Efficient in Spark

Renaming a column with withColumnRenamed() is considered a metadata-only operation in Spark.

Spark does not rewrite the underlying data immediately. Instead, it updates the logical execution plan and schema metadata.

Benefits:
- avoids expensive data rewriting
- reduces compute cost
- minimizes shuffle operations
- improves pipeline performance

# PART 11 — Data Modeling and Multi-Format Output

In [23]:
crime_fact = df_clean.select(
    col("Case Number"),
    col("District").alias("District_ID"),
    col("Arrest"),
    col("Domestic"),
    col("is_violent_crime"),
    col("days_since_crime")
)

crime_fact.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------+------+--------+----------------+----------------+
|Case Number|District_ID|Arrest|Domestic|is_violent_crime|days_since_crime|
+-----------+-----------+------+--------+----------------+----------------+
|    G000956|         19| false|   false|           false|            9276|
|    G000990|          3| false|   false|           false|            9276|
|    G001370|         14|  true|    true|            true|            9276|
|    G001742|          5| false|    true|           false|            9276|
|    G001998|          4|  true|   false|           false|            9276|
+-----------+-----------+------+--------+----------------+----------------+
only showing top 5 rows

In [24]:
district_dimension = df_clean.select(
    col("District").alias("District_ID"),
    col("district_name")
).dropDuplicates(["District_ID"])

district_dimension.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+--------------+
|District_ID| district_name|
+-----------+--------------+
|       NULL|       Unknown|
|          1|       Central|
|          2|     Wentworth|
|          3|Grand Crossing|
|          4| South Chicago|
|          5|       Calumet|
|          6|       Gresham|
|          7|     Englewood|
|          8|  Chicago Lawn|
|          9|       Deering|
+-----------+--------------+
only showing top 10 rows

In [25]:
s3_bucket = (
    "s3://<your-bucket>/"
    "shared/user_artur/"
)

district_dimension.write \
    .mode("overwrite") \
    .json(
        f"{s3_bucket}dimensions/district_dimension/"
    )

crime_fact.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(
        f"{s3_bucket}facts/crime_fact/"
    )

print(
    "Multi-format write completed successfully."
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Multi-format write completed successfully.

# JSON vs Parquet

# JSON
- row-based format
- human-readable
- flexible schema
- larger storage size
- slower analytical performance

# Parquet
- columnar storage format
- compressed and optimized for analytics
- faster query execution
- lower storage consumption
- efficient for Spark distributed workloads

Parquet is preferred because analytical queries often scan only specific columns instead of entire rows.

# PART 12 — Geospatial Analysis and Hotspot Detection

In [26]:
from pyspark.sql.functions import (
    col,
    concat,
    lit,
    round
)

df_geospatial = df_clean.withColumn(
    "geohash_cell",
    concat(
        round(col("Latitude"), 3).cast("string"),
        lit(","),
        round(col("Longitude"), 3).cast("string")
    )
)

df_hotspots = df_geospatial.groupBy(
    "geohash_cell"
).count().orderBy(
    col("count").desc()
).limit(5)

df_hotspots.show(truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------+-----+
|geohash_cell  |count|
+--------------+-----+
|41.976,-87.905|18123|
|41.882,-87.628|13482|
|41.755,-87.742|11204|
|41.884,-87.628|10664|
|41.898,-87.624|8460 |
+--------------+-----+

## Geospatial Grouping Explanation

Grouping directly by exact Latitude and Longitude is inefficient because coordinates are highly precise and many records become unique values.

Geohash-style grouping creates spatial buckets representing approximate geographic regions instead of exact points.

Benefits:
- enables hotspot detection
- reduces cardinality
- improves aggregation performance
- identifies spatial clusters more effectively

# PART 13 — Theoretical Review

# Architecture — Driver vs Executor

# Driver
The Driver is the central coordinator of a Spark application.

Responsibilities:
- creates the SparkSession
- builds the execution plan
- schedules tasks
- coordinates executors
- tracks job progress

# Executor
Executors are worker processes running on cluster nodes.

Responsibilities:
- execute Spark tasks
- process dataframe partitions
- perform transformations and actions
- return results to the Driver

# Which component executed df.count()?

The df.count() action was triggered by the Driver.
The actual distributed computation was executed by the Executors.

# Performance — Spark Stages and Join Boundaries

The final join operation created multiple Spark stages because joins typically require a shuffle operation.

At the join boundary:
- Spark redistributes data across executors based on the join key
- this redistribution is called a shuffle

A new stage starts because Spark must complete previous tasks before reshuffling and joining the data.

Joins are expensive because they involve:
- network transfer
- shuffle overhead
- executor synchronization
- possible disk spill

# Cloud Costs — On-Demand vs Spot Instances

For a daily batch ETL pipeline, Spot Instances are usually recommended for Worker nodes because they are significantly cheaper than On-Demand instances.

Advantages of Spot Instances:
- lower infrastructure cost
- suitable for fault-tolerant Spark workloads
- Spark can recover failed tasks automatically

Trade-Off:
- Spot Instances can be interrupted by AWS

Recommended strategy:
- use On-Demand for the Driver node
- use Spot Instances for Worker/Executor nodes

This hybrid approach balances reliability and cloud cost optimization.